In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
h = 5

w_range = np.arange(0.5, h - 1, 0.2)

In [ ]:
w = 1

In [ ]:
def analytic_scale(w, h):
    return (2 * (h - w) / np.pi + w) / h

### Experiment 1

In [ ]:
analytic_sfs = []
simulated_sfs = []
counter = 0
for w in w_range:
    print("w: ", w)
    print("analytic scale factor: ", analytic_scale(w, h))
    analytic_sfs.append(analytic_scale(w, h))
    
    res = 250 * w

    triArea = h * w / res
    avg_len = triArea ** 0.5

    import igl

    ipu = periodic_unit_helper.get_parallel_tube_ipu(h, w, res)

    fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

    # isheet.setRelaxedStiffnessEpsilon(1e-6)

    from tri_mesh_viewer import TriMeshViewer
    viewer = TriMeshViewer(ipu, width=768, height=640)
    viewer.showWireframe(True)
    ipu.sheet.rigidMotionPinVars

    fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

    # ipu.setVars(ipu.getVars() + fd_perturb)

    import time, vis
    ipu.sheet.setUseTensionFieldEnergy(False)
    ipu.sheet.setUseHessianProjectedEnergy(True)
    ipu.sheet.pressure = 1
    opts.niter = 200
    framerate = 5 # Update every 5 iterations
    def cb(it):
        if it % framerate == 0:
            viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
    
    render = viewer.offscreenRenderer(1000, 1000)
    render.render()
    render.save("inflated_with_width_{}.png".format(counter))
    counter += 1
    sfs = periodic_unit_helper.get_deformation_scale_factors(ipu)
    print("Simulated scale factor: ", sfs)
    simulated_sfs.append(sfs)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots()
plt.plot(w_range, analytic_sfs, '-o', label="Analytic scale factors")
plt.plot(w_range, simulated_sfs, '-o', label="Simulated scale factors")
ax.legend()
plt.xlabel("wall width")
plt.ylabel("scale factor")
plt.savefig('scale_factor_comparison.png', dpi=300)

### Experiment 2

In [ ]:
analytic_sfs = []
simulated_sfs = []
counter = 0
for w in w_range:
    print("w: ", w)
    print("analytic scale factor: ", analytic_scale(w, h))
    analytic_sfs.append(analytic_scale(w, h))
    
    import igl

    ipu = periodic_unit_helper.get_parallel_tube_periodic(h, w, 0.05)

    fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

    # isheet.setRelaxedStiffnessEpsilon(1e-6)

    from tri_mesh_viewer import TriMeshViewer
    viewer = TriMeshViewer(ipu, width=768, height=640)
    viewer.showWireframe(True)
    ipu.sheet.rigidMotionPinVars

    fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

    # ipu.setVars(ipu.getVars() + fd_perturb)

    import time, vis
    ipu.sheet.setUseTensionFieldEnergy(False)
    ipu.sheet.setUseHessianProjectedEnergy(True)
    ipu.sheet.pressure = 1
    opts.niter = 100
    framerate = 5 # Update every 5 iterations
    def cb(it):
        if it % framerate == 0:
            viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
    
    render = viewer.offscreenRenderer(1000, 1000)
    render.render()
    render.save("inflated_with_width_{}.png".format(counter))
    counter += 1
    sfs = periodic_unit_helper.get_deformation_scale_factors(ipu)
    print("Simulated scale factor: ", sfs)
    simulated_sfs.append(sfs)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots()
plt.plot(w_range, analytic_sfs, '-o', label="Analytic scale factors")
plt.plot(w_range, simulated_sfs, '-o', label="Simulated scale factors")
ax.legend()
plt.xlabel("wall width")
plt.ylabel("scale factor")
plt.savefig('scale_factor_comparison.png', dpi=300)